In [1]:
import numpy as np, pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import norm

In [36]:
class nptest:
  # methods:
  # test statistic and its truncated version
  # test power
  def  __init__(self, X, eps, f = None):
    self.X = X
    self.p = self.X.shape[0]
    self.c = np.sqrt(2*np.log(self.p))
    self.mu = (1+eps)*self.c

  def L(self, truncated = False, X_null = None):
    y = np.exp(self.X * self.mu - (self.mu ** 2)/2)
    if truncated:
      return np.mean(y*(self.X<self.c))
    else:
      if X_null is not None:
        y_null = np.exp(X_null * self.mu - (self.mu ** 2)/2)
        return(np.mean(y, axis = 0), np.mean(y_null, axis = 0))
      else:
        return(np.mean(y))


  def L_noeps(self, f):
    # N-P test statistic with mean defined by f
    if f is None:
      print("f not defined.")
      return None
    m = f(self.X)
    return np.mean(np.exp(self.X * m - m**2 / 2))


  def compute_quantiles_noeps(matrix, rep, f, q=0.95):
    if f is not None:
      l_vals = np.array([L_noeps(f) for i in range(rep)])
      return np.quantile(l_vals, q)

  def pwr(self, X_null):
      """
      NP power: proportion of repetitions where L on alternative
      exceeds corresponding L on null.
      """
      l_alt, l_null = self.L(X_null = X_null)
      return np.mean(l_alt > l_null)


In [30]:
 nptest(x, 0.2).pwr(X_null=X_null)

np.float64(0.0)

In [3]:
import random

random.seed(11)

In [4]:
def prob_different(l_vals, lTrunc_vals):
    return np.mean(l_vals != lTrunc_vals)

def summarize_l(vals):
    return {
    "mean": np.mean(vals),
    "max": np.max(vals),
    "quantile": np.quantile(vals, 0.95),
    "var":  np.var(vals, ddof=1),
}

def compute_quantiles(rep, eps, f, q = 0.95):
  runs = []
  for i in range(rep):
    x = np.random.normal(size = p, loc = 1+eps)
    npt = nptest(x, 1+eps)
    L_noeps = npt.L_noeps(lambda x: (1+eps)*np.sqrt(2*np.log(x.shape[0])))


In [5]:
rep = 5000
p = 5000
eps = -0.1
def run_simulations(p, eps, rep = 5000):
  runs_L = []
  runs_L_trunc = []
  for i in range(rep):
    x = np.random.normal(size = p)
    #print(x.mean())
    npt = nptest(x, eps)
    runs_L.append(npt.L())
    runs_L_trunc.append(npt.L(truncated = True))
  return(np.array(runs_L), np.array(runs_L_trunc))

In [72]:
for eps in [-0.1, -0.2, -0.3]:
  for p in [5000, 50000, 500000]:
    norm_cdf = norm.cdf(-eps*np.sqrt(2*np.log(p)))
    print(f"p: {p}, eps: {eps}, norm_cdf: {norm_cdf}")

p: 5000, eps: -0.1, norm_cdf: 0.660096807151621
p: 50000, eps: -0.1, norm_cdf: 0.6790999258232608
p: 500000, eps: -0.1, norm_cdf: 0.6957780937788962
p: 5000, eps: -0.2, norm_cdf: 0.795443253456883
p: 50000, eps: -0.2, norm_cdf: 0.8239093822824707
p: 500000, eps: -0.2, norm_cdf: 0.8472221645572742
p: 5000, eps: -0.3, norm_cdf: 0.8921757160633608
p: 50000, eps: -0.3, norm_cdf: 0.9185749915354582
p: 500000, eps: -0.3, norm_cdf: 0.9378396579277866


In [7]:
for eps in [-0.1, -0.2, -0.3]:
  for p in [5000, 50000, 500000]:
    runs_L, runs_L_trunc = run_simulations(p, eps)
    print(f"p={p}, eps={eps}, probability L, L_trunc differ={prob_different(np.array(runs_L),np.array(runs_L_trunc))}")
    print(summarize_l(runs_L), summarize_l(runs_L_trunc))


p=5000, eps=-0.1, probability L, L_trunc differ=0.0924
{'mean': np.float64(1.0312810341939302), 'max': np.float64(305.21010815793846), 'quantile': np.float64(2.3025592317745915), 'var': np.float64(23.842429763776437)} {'mean': np.float64(0.6482449895404263), 'max': np.float64(2.5833968870553967), 'quantile': np.float64(1.2373465506296264), 'var': np.float64(0.08870958180808254)}
p=50000, eps=-0.1, probability L, L_trunc differ=0.0792
{'mean': np.float64(1.007638307857342), 'max': np.float64(103.76545377021381), 'quantile': np.float64(2.1759699311697775), 'var': np.float64(7.498207752671676)} {'mean': np.float64(0.6835113807605181), 'max': np.float64(2.2210883014561267), 'quantile': np.float64(1.2628267877357977), 'var': np.float64(0.07927776408484734)}
p=500000, eps=-0.1, probability L, L_trunc differ=0.0664
{'mean': np.float64(0.9229100398346203), 'max': np.float64(123.5446601470986), 'quantile': np.float64(1.8898009620406166), 'var': np.float64(4.939126242090483)} {'mean': np.float64

from Candes (2023), we know that the probability that the $L$ and its truncated version differ $\rightarrow 0$ with respect to $p$, under $H_0$. We also know that the variance of $L_{truncated}$ converges to $o(1)$ and its mean reaches $\Phi(-\epsilon\sqrt(2*\log p))$, which is confirmed above.

In [9]:
def simulate_critical(p, eps, rep = 8000):
  """ simulate critical value of the NP test for the needle in haystack problem"""
  runs_L = []
  for i in range(rep):
    x = np.random.normal(size = p)
    npt = nptest(x, eps)
    runs_L.append(npt.L())
  return(np.quantile(np.array(runs_L), 0.95))

In [10]:
X_small_08 = simulate_critical(5000, -0.2)
X_large_08 = simulate_critical(50000, -0.2)
print(f"p=5000, eps=-0.2: {X_small_08}, p=50000: {X_large_08}")

p=5000, eps=-0.2: 1.93329971760327, p=50000: 1.8106725134754402


In [11]:
X_small_12 = simulate_critical(5000, 0.2)
X_large_12= simulate_critical(50000, 0.2)
print(f"p=5000, eps=0.2: {X_small_12}, p=50000: {X_large_12}")

p=5000, eps=0.2: 1.5837267315433516, p=50000: 1.4206796945782565


In [70]:
class bonftest:
  """
  implementation of the Bonferroni correction.
  methods:
  p-values of the individual tests, from which we want to test the global null
  test power
  """
  def  __init__(self, X, eps, f = None):
    self.X = X
    self.p = self.X.shape[0]
    self.c = np.sqrt(2*np.log(self.p))
    self.mu = (1+eps)*self.c

  def _pval(self):
    return 2*norm.cdf(-np.abs(self.X))

  def pwr(self, alpha = 0.05):
    pvals = self._pval()
    return np.mean(np.min(pvals, axis=0) <= alpha / self.p)


From Candes' materials, we know that the Bonferroni test in the "needle in a haystack problem" has a detection threshold: if the signal size is larger than $\sqrt(2\log n)$, i.e. we have $\mu^{i}=(1+\epsilon)\sqrt(2\log n)$, it will have power of 1 in the limit with respect to the number of individual hypotheses. Conversely, if $\mu^{i}=(1-\epsilon)\sqrt(2\log n)$, the test will be powerless in the limit. An example of this is shown below.

In [67]:

import numpy as np
def generate_X(p, rep, mi):
    """Generate a (p x rep) matrix where one samples (the first) is drawn from N(mi, 1), the rest from N(0, 1)."""
    X       = np.random.normal(size=(p, rep))
    X[0, :] = np.random.normal(loc=mi, scale=1, size=rep)
    return X



p_values = [p, pp]
epsilons = [0.2, -0.2]


p      = 5000
pp     = 50000
alpha  = 0.05
repNum = 750

# compare the powers of NP test and the test with Bonferroni correction.

def compute_mu(eps, p):
    return (1 + eps) * np.sqrt(2 * np.log(p))

def compare_pwrs(p, eps, alpha = 0.05, repNum = 2000):
  results = {}
  mu = compute_mu(eps, p)
  X_null = np.random.normal(size=(p, repNum))
  X      = generate_X(p, repNum, mu)
  label = str(1+eps)
  b = bonftest(X, eps)
  npt = nptest(X, eps)
  results[f"NP {label}, p = {p}"]   = npt.pwr(X_null = X_null)
  results[f"Bonf {label}, p = {p}"] = b.pwr()
  return(results)

compare_pwrs(p, 0.2), compare_pwrs(p, -0.2), compare_pwrs(pp, 0.2), compare_pwrs(pp, -0.2)

({'NP 1.2, p = 5000': np.float64(0.9375),
  'Bonf 1.2, p = 5000': np.float64(0.7275)},
 {'NP 0.8, p = 5000': np.float64(0.663),
  'Bonf 0.8, p = 5000': np.float64(0.179)},
 {'NP 1.2, p = 50000': np.float64(0.941),
  'Bonf 1.2, p = 50000': np.float64(0.776)},
 {'NP 0.8, p = 50000': np.float64(0.645),
  'Bonf 0.8, p = 50000': np.float64(0.1665)})